<div style="background:linear-gradient(135deg, rgba(249,112,102,0.18), rgba(249,112,102,0.02)); border-left:6px solid #F97066; border-radius:10px; padding:20px 24px; margin-bottom:20px;">
<h1 style="margin:0; color:#F97066; font-size:1.8em;">🤝 Sistemas Multiagente</h1>
<p style="margin:6px 0 0; opacity:0.8;">Unidad 5 — Cuando varios agentes especializados coordinan mejor que uno solo generalista</p>
</div>

En <code>3-multiples-herramientas.ipynb</code> se señaló que un agente con demasiadas herramientas distintas elige peor entre ellas. Una alternativa es dividir el trabajo: varios agentes <strong>especialistas</strong>, cada uno con pocas herramientas de un mismo dominio, coordinados por un agente <strong>supervisor</strong> que decide a quién delegar cada parte de la tarea.

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0 24px; background:rgba(14,165,233,0.04);">
<strong>📑 Contenido de esta guía</strong>
<ol style="margin:8px 0 0; padding-left:20px;">
<li><a href="#patron-agentes-como-herramientas">El patrón: agentes como herramientas</a></li>
<li><a href="#especialistas">Dos agentes especialistas</a></li>
<li><a href="#supervisor">El agente supervisor</a></li>
<li><a href="#cierre">Cierre y próximos pasos</a></li>
</ol>
</div>

<a id="patron-agentes-como-herramientas"></a>

## <span style="color:#F97066;">El patrón: agentes como herramientas</span>

No existe una única forma de construir un sistema multiagente. El patrón más simple — y el que usa este notebook — es <strong>agentes como herramientas</strong>: cada agente especialista se envuelve en una función <code>@tool</code>, exactamente igual que la herramienta RAG del notebook anterior. Para el agente supervisor, "consultar al especialista en finanzas" es una herramienta más, aunque por dentro esa herramienta ejecute un agente completo con su propio ciclo ReAct.

Esta simplicidad es su principal ventaja: no se necesita ningún mecanismo nuevo más allá de lo ya visto — <code>create_agent</code> y <code>@tool</code> — para coordinar varios agentes.

<a id="especialistas"></a>

## <span style="color:#F97066;">Dos agentes especialistas</span>

Se definen dos especialistas con dominios distintos y sin solapamiento: uno de cálculos financieros, otro de análisis de texto. Cada uno recibe solo las herramientas de su propio dominio.

In [1]:
import os
import logging
from dotenv import load_dotenv

load_dotenv()
logging.getLogger("google_genai.models").setLevel(logging.ERROR)

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain.tools import tool

llm = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite", api_key=os.getenv("GOOGLE_API_KEY"))


@tool
def convertir_moneda(monto: float, tasa_cambio: float) -> str:
    """Convierte un monto de una moneda a otra usando una tasa de cambio dada.

    Args:
        monto: La cantidad a convertir, en la moneda de origen.
        tasa_cambio: Cuántas unidades de la moneda destino equivalen a una unidad de la moneda de origen.
    """
    return f"{monto * tasa_cambio:.2f}"


agente_finanzas = create_agent(
    model=llm,
    tools=[convertir_moneda],
    system_prompt="Eres un especialista en cálculos financieros y conversión de moneda. Responde de forma breve.",
)


@tool
def contar_palabras(texto: str) -> str:
    """Cuenta el número de palabras en un texto.

    Args:
        texto: El texto a analizar.
    """
    return str(len(texto.split()))


agente_texto = create_agent(
    model=llm,
    tools=[contar_palabras],
    system_prompt="Eres un especialista en análisis de texto. Responde de forma breve.",
)

print("Dos agentes especialistas construidos correctamente.")

Dos agentes especialistas construidos correctamente.


<a id="supervisor"></a>

## <span style="color:#F97066;">El agente supervisor</span>

Cada especialista se envuelve en una herramienta que lo invoca y devuelve su respuesta final. El supervisor no ve las herramientas internas de cada especialista (<code>convertir_moneda</code>, <code>contar_palabras</code>) — solo ve dos herramientas de alto nivel: "consultar al especialista en finanzas" y "consultar al especialista en texto".

In [2]:
@tool
def consultar_especialista_finanzas(pregunta: str) -> str:
    """Consulta al agente especialista en cálculos financieros y conversión de moneda.

    Args:
        pregunta: La pregunta o tarea financiera a resolver.
    """
    resultado = agente_finanzas.invoke({"messages": [{"role": "user", "content": pregunta}]})
    return resultado["messages"][-1].text


@tool
def consultar_especialista_texto(pregunta: str) -> str:
    """Consulta al agente especialista en análisis de texto.

    Args:
        pregunta: La pregunta o tarea de análisis de texto a resolver.
    """
    resultado = agente_texto.invoke({"messages": [{"role": "user", "content": pregunta}]})
    return resultado["messages"][-1].text


supervisor = create_agent(
    model=llm,
    tools=[consultar_especialista_finanzas, consultar_especialista_texto],
    system_prompt=(
        "Coordinas dos especialistas: uno de finanzas y uno de análisis de texto. "
        "Delega cada parte de la pregunta al especialista correcto y combina sus respuestas "
        "en una respuesta final coherente."
    ),
)

print("Agente supervisor construido correctamente.")

Agente supervisor construido correctamente.


In [3]:
consulta = (
    "Convierte 200 dólares a pesos colombianos con una tasa de cambio de 4000, "
    "y además dime cuántas palabras tiene la frase: los agentes trabajan en equipo."
)

resultado = supervisor.invoke({"messages": [{"role": "user", "content": consulta}]})

for mensaje in resultado["messages"]:
    if getattr(mensaje, "tool_calls", None):
        for llamada in mensaje.tool_calls:
            print(f"→ Supervisor delega en: {llamada['name']}({llamada['args']})")

print("\nRespuesta final:\n")
print(resultado["messages"][-1].text)

→ Supervisor delega en: consultar_especialista_finanzas({'pregunta': 'Convierte 200 dólares a pesos colombianos con una tasa de cambio de 4000'})
→ Supervisor delega en: consultar_especialista_texto({'pregunta': '¿Cuántas palabras tiene la frase: los agentes trabajan en equipo?'})

Respuesta final:

Aquí tienes los resultados de ambas solicitudes:

* **Conversión financiera:** 200 dólares equivalen a **800,000 pesos colombianos** (calculado con una tasa de cambio de 4,000 COP por dólar).
* **Análisis de texto:** La frase "los agentes trabajan en equipo" tiene **5 palabras**.


La pregunta combina dos tareas de dominios distintos, y el supervisor las repartió correctamente: delegó la conversión de moneda al especialista financiero y el conteo de palabras al especialista de texto, y combinó ambas respuestas en una sola. Ningún código explícito le dijo "esta parte es de finanzas, esta otra es de texto" — esa división también la decidió el LLM, igual que la elección de herramientas en <code>3-multiples-herramientas.ipynb</code>.

<div style="border-left:4px solid #6366F1; background:rgba(99,102,241,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>📝 Nota</strong><br>
Este patrón escala mejor que un solo agente con todas las herramientas: cada especialista solo necesita razonar sobre las pocas herramientas de su propio dominio, y el supervisor solo necesita decidir <em>a quién</em> preguntar, no <em>cómo</em> resolver cada tarea. El costo es la latencia: cada consulta a un especialista es, en sí misma, una llamada completa al LLM (o varias, si el especialista usa herramientas), así que una pregunta que involucra a dos especialistas termina haciendo varias llamadas encadenadas.
</div>

---

<a id="cierre"></a>

# <span style="color:#F97066;">🎯 Cierre y próximos pasos</span>

<div style="border-left:4px solid #14B8A6; background:rgba(20,184,166,0.08); border-radius:6px; padding:10px 16px; margin:16px 0;">
<strong>✅ Resumen</strong><br>
Este notebook mostró un sistema multiagente con el patrón agentes-como-herramientas:

- Cada agente especialista tiene solo las herramientas de su propio dominio, lo que facilita que elija bien entre ellas.
- Un agente supervisor coordina a los especialistas envolviendo cada uno como una herramienta más.
- El supervisor decide, sin lógica programada a mano, a qué especialista delegar cada parte de una pregunta compuesta.
- El costo de este patrón es la latencia: cada especialista consultado implica una o más llamadas adicionales al LLM.

</div>

<div style="border:1px solid rgba(14,165,233,0.35); border-radius:10px; padding:14px 20px; margin:16px 0; background:rgba(14,165,233,0.04);">
<strong>➡️ Continúe con</strong>
<ul style="margin:8px 0 0; padding-left:20px;">
<li><code>7-evaluacion-de-agentes.ipynb</code> — cómo medir si un agente elige bien sus herramientas y responde correctamente.</li>
</ul>
</div>